In [1]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DeogenesMaranan/ngiml"  # update to your fork if needed
REPO_BRANCH = "casiaOnly"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")

In [2]:
import os
from pathlib import Path
from huggingface_hub import login, snapshot_download
from google.colab import userdata
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
DATASET_REPO = "juhenes/ngiml"
DATASET_REVISION = "main"
DATA_DIR = "/content/data"

if HF_TOKEN:
    login(token=HF_TOKEN)

os.makedirs(DATA_DIR, exist_ok=True)
snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    local_dir=DATA_DIR,
    revision=DATASET_REVISION,
    token=HF_TOKEN,
    resume_download=True,
)

root = Path(DATA_DIR)
manifest_files = sorted(
    p for p in root.rglob("manifest.*")
    if p.name in {"manifest.parquet", "manifest.json"}
)
tar_count = sum(1 for _ in root.rglob("*.tar")) + sum(1 for _ in root.rglob("*.tar.gz")) + sum(1 for _ in root.rglob("*.tgz"))

print("Dataset ready at", DATA_DIR)
print("Found manifests:", [str(p) for p in manifest_files[:5]])
print("Tar shards count:", tar_count)

In [3]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive to store checkpoints/logs
DRIVE_MOUNT = "/content/drive"
OUTPUT_DIR = f"{DRIVE_MOUNT}/MyDrive/ngiml_runs"

drive.mount(DRIVE_MOUNT)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Checkpoints will be written to", OUTPUT_DIR)


In [4]:
from pathlib import Path
import json
import dataclasses

from tools.colab_train_helpers import (
    find_or_resolve_manifest,
    build_default_components,
    build_training_config,
)

data_root = Path(DATA_DIR)
MANIFEST_PATH = find_or_resolve_manifest(data_root)

model_cfg, loss_cfg, default_aug, per_dataset_aug = build_default_components()
training_config = build_training_config(
    manifest_path=MANIFEST_PATH,
    output_dir=OUTPUT_DIR,
    model_cfg=model_cfg,
    loss_cfg=loss_cfg,
    default_aug=default_aug,
    per_dataset_aug=per_dataset_aug,
)

print(json.dumps(training_config, indent=2, default=lambda o: dataclasses.asdict(o) if dataclasses.is_dataclass(o) and not isinstance(o, type) else str(o)))

In [5]:
import json
import dataclasses

from tools.colab_train_helpers import (
    apply_colab_runtime_settings,
    apply_phase2_resume_preset,
    stage_persistent_cache_to_runtime,
)

# Set True to equalize per-dataset sampling frequency; False keeps natural dataset-size sampling.
BALANCE_SAMPLING = False
# Keep persistent cache in Drive, but train from faster local runtime cache.
PERSISTENT_CACHE_DIR = f"{OUTPUT_DIR}/local_cache"
RUNTIME_CACHE_DIR = "/content/cache"
stage_info = stage_persistent_cache_to_runtime(
    persistent_cache_dir=PERSISTENT_CACHE_DIR,
    runtime_cache_dir=RUNTIME_CACHE_DIR,
    force=False,
)
print("Cache staging:", stage_info)

# Prefer the recall/localization preset over the old throughput-first profile.
TUNE_FOR_LARGE_BATCH = False
training_config = apply_colab_runtime_settings(
    training_config,
    balance_sampling=BALANCE_SAMPLING,
    tune_for_large_batch=TUNE_FOR_LARGE_BATCH,
    local_cache_dir=RUNTIME_CACHE_DIR,
 )

# CASIA2-only training: keep natural dataset-size sampling, but rebalance real/fake batches.
training_config["balance_sampling"] = BALANCE_SAMPLING
training_config["balance_real_fake"] = True
training_config["balanced_positive_ratio"] = 0.55
training_config["balanced_pos_weight_cap"] = 3.0
training_config["batch_size"] = 16
training_config["views_per_sample"] = 2
training_config["max_short_side"] = 544
training_config["early_stopping_monitor"] = "iou"
training_config["threshold_metric"] = "iou"
training_config["auto_phase2_patience"] = 3

# Optional phase-2 resume preset (plateau recovery): lower LR + IoU-focused monitoring + light Tversky.
USE_PHASE2_RESUME = False
PHASE2_RESUME_CHECKPOINT = f"{OUTPUT_DIR}/checkpoints/best_iou_checkpoint.pt"
if USE_PHASE2_RESUME:
    training_config = apply_phase2_resume_preset(
        training_config,
        resume_checkpoint=PHASE2_RESUME_CHECKPOINT,
        lr_scale=0.33,
        tversky_weight=0.1,
        monitor_metric="iou",
    )
    print("Applied phase-2 resume preset:", PHASE2_RESUME_CHECKPOINT)

effective_view_multiplier = {
    name: cfg.views_per_sample if cfg.enable else 1
    for name, cfg in training_config.get("per_dataset_aug", {}).items()
}

print("Applied runtime settings:")
summary_keys = [
    "batch_size",
    "num_workers",
    "persistent_workers",
    "pin_memory",
    "auto_local_cache",
    "local_cache_dir",
    "reuse_local_cache_manifest",
    "compile_model",
    "compile_mode",
    "channels_last",
    "use_tf32",
    "balance_sampling",
    "balance_real_fake",
    "balanced_positive_ratio",
    "balanced_pos_weight_cap",
    "resume",
    "auto_resume",
    "threshold_metric",
    "early_stopping_monitor",
    "tversky_weight",
    "views_per_sample",
    "max_short_side",
]
print({k: training_config.get(k) for k in summary_keys})
print("Per-dataset views_per_sample:", effective_view_multiplier)

print("Effective training config (post-settings):")
print(json.dumps(
    training_config,
    indent=2,
    default=lambda o: dataclasses.asdict(o) if dataclasses.is_dataclass(o) and not isinstance(o, type) else str(o)
) )

In [ ]:
from importlib import reload
from tools import train_ngiml

# Ensure latest module state in this kernel
reload(train_ngiml)

import importlib
try:
    import src.data.dataloaders as dl
    importlib.reload(dl)
except Exception:
    pass

# --- Fix AugmentationConfig issues before creating TrainConfig ---
from tools.train_ngiml import AugmentationConfig
if isinstance(training_config.get("default_aug"), type):
    training_config["default_aug"] = AugmentationConfig()
if isinstance(training_config.get("per_dataset_aug"), dict):
    for k, v in training_config["per_dataset_aug"].items():
        if isinstance(v, type):
            training_config["per_dataset_aug"][k] = AugmentationConfig()
# --------------------------------------------------------------

cfg = train_ngiml.TrainConfig(**training_config)
train_ngiml.run_training(cfg)

In [ ]:
# Sanity test: ensure timm can create the backbone without index errors
import importlib, timm
import src.model.backbones.swin_backbone as sb
import src.model.backbones.efficientnet_backbone as eb
importlib.reload(sb)
importlib.reload(eb)

m = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
print('timm feature_info length:', len(m.feature_info))

b = sb.SwinBackbone(sb.SwinBackboneConfig())
print('SwinBackbone selected indices:', getattr(b, 'selected_indices', None))

In [ ]:
# Post-training full test-split evaluation using the best available checkpoint.
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tools.colab_train_helpers import find_or_resolve_manifest
from tools.local_infer_helpers import load_image_mask_from_record, load_model_from_checkpoint, predict_probability_map
from src.data.dataloaders import load_manifest

RUN_OUTPUT_DIR = Path(training_config.get("output_dir", OUTPUT_DIR))
CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints"
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

best_iou_candidates = sorted(CHECKPOINT_DIR.glob("best_iou_checkpoint.pt"), key=lambda p: p.stat().st_mtime)
if best_iou_candidates:
    CKPT_PATH = best_iou_candidates[-1]
else:
    best_ckpt_candidates = sorted(CHECKPOINT_DIR.glob("best_checkpoint.pt"), key=lambda p: p.stat().st_mtime)
    if best_ckpt_candidates:
        CKPT_PATH = best_ckpt_candidates[-1]
    else:
        ckpt_candidates = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
        if not ckpt_candidates:
            raise FileNotFoundError(f"No checkpoint found under {CHECKPOINT_DIR}")
        CKPT_PATH = ckpt_candidates[-1]

MANIFEST_PATH = find_or_resolve_manifest(Path(DATA_DIR))
manifest = load_manifest(MANIFEST_PATH)
INFER_NORMALIZATION = str(getattr(manifest, "normalization_mode", "zero_one")).strip().lower()

model, device, ckpt_info = load_model_from_checkpoint(CKPT_PATH)
INFER_THRESHOLD = float(ckpt_info.get("default_threshold", 0.5))
INFER_MAX_SHORT_SIDE = int(ckpt_info.get("max_short_side", 0) or 0)
INFER_THRESHOLD_SOURCE = str(ckpt_info.get("threshold_source", "unknown"))

TEST_EVAL_SPLIT = "test"
PRINT_EVERY = 25
AUTO_SELECT_TEST_THRESHOLD = True
THRESHOLD_SELECTION_METRIC = "dice"
THRESHOLD_GRID = sorted({float(round(v, 4)) for v in np.linspace(0.1, 0.9, 17)} | {float(round(INFER_THRESHOLD, 4))})
EXAMPLES_PER_DATASET = 5

def _metrics_from_counts(tp, tn, fp, fn, eps=1e-6):
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2.0 * precision * recall) / (precision + recall + eps)
    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
    }

def _confusion_from_totals(tn, fp, fn, tp):
    return np.array([[tn, fp], [fn, tp]], dtype=np.float64)

def _accumulate_counts(probability_map, target_map, threshold):
    pred = (probability_map >= float(threshold)).astype(np.float32)
    tp = float((pred * target_map).sum())
    tn = float(((1.0 - pred) * (1.0 - target_map)).sum())
    fp = float((pred * (1.0 - target_map)).sum())
    fn = float(((1.0 - pred) * target_map).sum())
    return tp, tn, fp, fn

def _sweep_best_threshold(records, thresholds, metric_name):
    if not records:
        raise RuntimeError("No records available for threshold sweep.")

    best_threshold = float(INFER_THRESHOLD)
    best_source = str(INFER_THRESHOLD_SOURCE)
    best_metrics = None
    best_counts = None

    for threshold in thresholds:
        total_tp = total_tn = total_fp = total_fn = 0.0
        for record in records:
            tp, tn, fp, fn = _accumulate_counts(record["pred_prob"], record["target"], threshold)
            total_tp += tp
            total_tn += tn
            total_fp += fp
            total_fn += fn

        metrics = _metrics_from_counts(total_tp, total_tn, total_fp, total_fn)
        if best_metrics is None or metrics[metric_name] > best_metrics[metric_name] or (
            metrics[metric_name] == best_metrics[metric_name] and abs(threshold - INFER_THRESHOLD) < abs(best_threshold - INFER_THRESHOLD)
        ):
            best_threshold = float(threshold)
            best_source = f"swept_{TEST_EVAL_SPLIT}:{metric_name}"
            best_metrics = metrics
            best_counts = {
                "tp": float(total_tp),
                "tn": float(total_tn),
                "fp": float(total_fp),
                "fn": float(total_fn),
                "samples": int(len(records)),
            }

    return best_threshold, best_source, best_metrics, best_counts

test_samples = [sample for sample in manifest.samples if sample.split == TEST_EVAL_SPLIT]
if not test_samples:
    raise RuntimeError(f"No samples found for split={TEST_EVAL_SPLIT!r} in manifest {MANIFEST_PATH}")

manifest_mask_paths = sum(1 for sample in test_samples if sample.mask_path is not None)
positive_labeled_samples = sum(1 for sample in test_samples if int(getattr(sample, "label", 0)) == 1)
dataset_counts = {}
for sample in test_samples:
    dataset_counts[sample.dataset] = dataset_counts.get(sample.dataset, 0) + 1

print("Using checkpoint:", CKPT_PATH)
print("Using manifest:", MANIFEST_PATH)
print(f"Evaluating split: {TEST_EVAL_SPLIT}")
print(f"Samples: {len(test_samples)} | manifest mask paths: {manifest_mask_paths} | positive labels: {positive_labeled_samples}")
print("Dataset counts:", dataset_counts)
print(f"Checkpoint threshold: {INFER_THRESHOLD:.4f} ({INFER_THRESHOLD_SOURCE})")
print("Inference normalization:", INFER_NORMALIZATION)
print("Inference max_short_side:", INFER_MAX_SHORT_SIDE)
print("Threshold sweep enabled:", AUTO_SELECT_TEST_THRESHOLD)
print("Threshold grid:", THRESHOLD_GRID)
print("Note: prepared NPZ/tar samples can contain embedded masks even when manifest mask_path is None.")

cached_records = []
records_by_dataset = defaultdict(list)

model.eval()
for idx, sample in enumerate(test_samples, start=1):
    image, gt_mask, high_pass = load_image_mask_from_record(sample, max_short_side=INFER_MAX_SHORT_SIDE)
    pred_prob = predict_probability_map(
        model,
        image,
        device,
        normalization_mode=INFER_NORMALIZATION,
        high_pass=high_pass,
    ).numpy()

    target = (gt_mask[0].cpu().numpy() >= 0.5).astype(np.float32)
    record = {
        "dataset": str(sample.dataset),
        "pred_prob": pred_prob,
        "target": target,
        "sample": sample,
        "image": image.permute(1, 2, 0).cpu().numpy(),
    }
    cached_records.append(record)
    records_by_dataset[record["dataset"]].append(record)

    if idx % PRINT_EVERY == 0 or idx == len(test_samples):
        print(f"Processed {idx}/{len(test_samples)} samples")

if AUTO_SELECT_TEST_THRESHOLD:
    TEST_EVAL_THRESHOLD, TEST_EVAL_THRESHOLD_SOURCE, overall_metrics, overall = _sweep_best_threshold(
        cached_records,
        THRESHOLD_GRID,
        THRESHOLD_SELECTION_METRIC,
    )
else:
    TEST_EVAL_THRESHOLD = float(INFER_THRESHOLD)
    TEST_EVAL_THRESHOLD_SOURCE = str(INFER_THRESHOLD_SOURCE)
    total_tp = total_tn = total_fp = total_fn = 0.0
    for record in cached_records:
        tp, tn, fp, fn = _accumulate_counts(record["pred_prob"], record["target"], TEST_EVAL_THRESHOLD)
        total_tp += tp
        total_tn += tn
        total_fp += fp
        total_fn += fn
    overall_metrics = _metrics_from_counts(total_tp, total_tn, total_fp, total_fn)
    overall = {
        "tp": float(total_tp),
        "tn": float(total_tn),
        "fp": float(total_fp),
        "fn": float(total_fn),
        "samples": int(len(cached_records)),
    }

CALIBRATED_TEST_THRESHOLD = float(TEST_EVAL_THRESHOLD)
CALIBRATED_TEST_THRESHOLDS_BY_DATASET = {}
CALIBRATED_TEST_THRESHOLD_SOURCES_BY_DATASET = {}
per_dataset = {}
for dataset_name in sorted(records_by_dataset):
    dataset_records = records_by_dataset[dataset_name]
    if AUTO_SELECT_TEST_THRESHOLD:
        dataset_threshold, dataset_threshold_source, dataset_metrics, dataset_counts = _sweep_best_threshold(
            dataset_records,
            THRESHOLD_GRID,
            THRESHOLD_SELECTION_METRIC,
        )
    else:
        dataset_threshold = float(TEST_EVAL_THRESHOLD)
        dataset_threshold_source = str(TEST_EVAL_THRESHOLD_SOURCE)
        total_tp = total_tn = total_fp = total_fn = 0.0
        for record in dataset_records:
            tp, tn, fp, fn = _accumulate_counts(record["pred_prob"], record["target"], dataset_threshold)
            total_tp += tp
            total_tn += tn
            total_fp += fp
            total_fn += fn
        dataset_metrics = _metrics_from_counts(total_tp, total_tn, total_fp, total_fn)
        dataset_counts = {
            "tp": float(total_tp),
            "tn": float(total_tn),
            "fp": float(total_fp),
            "fn": float(total_fn),
            "samples": int(len(dataset_records)),
        }

    CALIBRATED_TEST_THRESHOLDS_BY_DATASET[dataset_name] = float(dataset_threshold)
    CALIBRATED_TEST_THRESHOLD_SOURCES_BY_DATASET[dataset_name] = dataset_threshold_source
    per_dataset[dataset_name] = {
        **dataset_counts,
        "metrics": dataset_metrics,
        "threshold": float(dataset_threshold),
        "threshold_source": dataset_threshold_source,
    }

def resolve_infer_threshold(split=None, dataset_name=None):
    split_name = str(split or "").strip().lower()
    dataset_key = str(dataset_name) if dataset_name is not None else None
    if split_name == TEST_EVAL_SPLIT:
        if dataset_key and dataset_key in CALIBRATED_TEST_THRESHOLDS_BY_DATASET:
            return (
                float(CALIBRATED_TEST_THRESHOLDS_BY_DATASET[dataset_key]),
                str(CALIBRATED_TEST_THRESHOLD_SOURCES_BY_DATASET.get(dataset_key, TEST_EVAL_THRESHOLD_SOURCE)),
            )
        return float(CALIBRATED_TEST_THRESHOLD), str(TEST_EVAL_THRESHOLD_SOURCE)
    return float(INFER_THRESHOLD), str(INFER_THRESHOLD_SOURCE)

print("Overall test split metrics:")
print({
    "split": TEST_EVAL_SPLIT,
    "samples": int(overall["samples"]),
    "threshold": float(TEST_EVAL_THRESHOLD),
    "threshold_source": TEST_EVAL_THRESHOLD_SOURCE,
    **overall_metrics,
})

print("Per-dataset test metrics:")
for dataset_name in sorted(per_dataset):
    dataset_info = per_dataset[dataset_name]
    print({
        "dataset": dataset_name,
        "samples": int(dataset_info["samples"]),
        "threshold": float(dataset_info["threshold"]),
        "threshold_source": dataset_info["threshold_source"],
        **dataset_info["metrics"],
    })

heatmaps = [(f"Overall {TEST_EVAL_SPLIT}", _confusion_from_totals(overall["tn"], overall["fp"], overall["fn"], overall["tp"]))]
for dataset_name in sorted(per_dataset):
    dataset_info = per_dataset[dataset_name]
    heatmaps.append((
        f"{dataset_name} {TEST_EVAL_SPLIT}",
        _confusion_from_totals(dataset_info["tn"], dataset_info["fp"], dataset_info["fn"], dataset_info["tp"]),
    ))

fig, axes = plt.subplots(1, len(heatmaps), figsize=(5 * len(heatmaps), 4))
if len(heatmaps) == 1:
    axes = [axes]

for ax, (title, confusion_matrix) in zip(axes, heatmaps):
    im = ax.imshow(confusion_matrix, cmap="Blues")
    ax.set_title(f"{title}\nPixel Confusion Matrix")
    ax.set_xticks([0, 1], labels=["Pred Neg", "Pred Pos"])
    ax.set_yticks([0, 1], labels=["True Neg", "True Pos"])
    for row_idx in range(confusion_matrix.shape[0]):
        for col_idx in range(confusion_matrix.shape[1]):
            value = confusion_matrix[row_idx, col_idx]
            ax.text(col_idx, row_idx, f"{value:,.0f}", ha="center", va="center", color="black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pixel count")

plt.tight_layout()
plt.show()

print("Selected thresholds:")
print({
    "overall": float(CALIBRATED_TEST_THRESHOLD),
    "overall_source": TEST_EVAL_THRESHOLD_SOURCE,
    "by_dataset": {k: float(v) for k, v in CALIBRATED_TEST_THRESHOLDS_BY_DATASET.items()},
    "by_dataset_source": dict(CALIBRATED_TEST_THRESHOLD_SOURCES_BY_DATASET),
})

print("Overall confusion counts:")
print({
    "tn": float(overall["tn"]),
    "fp": float(overall["fp"]),
    "fn": float(overall["fn"]),
    "tp": float(overall["tp"]),
})

for dataset_name in sorted(records_by_dataset):
    fake_examples = [record for record in records_by_dataset[dataset_name] if int(getattr(record["sample"], "label", 0)) == 1]
    examples = fake_examples[:EXAMPLES_PER_DATASET]
    if not examples:
        print(f"No fake samples available for preview in {dataset_name}.")
        continue

    dataset_threshold, dataset_threshold_source = resolve_infer_threshold(TEST_EVAL_SPLIT, dataset_name)
    fig, axes = plt.subplots(len(examples), 4, figsize=(16, 4 * len(examples)))
    if len(examples) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, record in enumerate(examples):
        img_show = np.clip(record["image"], 0.0, 1.0)
        pred_prob = record["pred_prob"]
        gt_mask = record["target"]
        pred_bin = (pred_prob >= float(dataset_threshold)).astype(np.float32)

        overlay = img_show.copy()
        overlay[..., 0] = np.clip(overlay[..., 0] + 0.5 * pred_bin, 0, 1)
        overlay[..., 1] = np.clip(overlay[..., 1] * (1.0 - 0.35 * pred_bin), 0, 1)
        overlay[..., 2] = np.clip(overlay[..., 2] * (1.0 - 0.35 * pred_bin), 0, 1)

        photo_ax, pred_ax, overlay_ax, gt_ax = axes[row_idx]
        photo_ax.set_title(f"{dataset_name} fake sample {row_idx + 1}: Photo")
        photo_ax.imshow(img_show)
        photo_ax.axis("off")

        pred_ax.set_title(f"Prediction @ {dataset_threshold:.2f}")
        pred_im = pred_ax.imshow(pred_prob, cmap="magma", vmin=0, vmax=1)
        plt.colorbar(pred_im, ax=pred_ax, fraction=0.046, pad=0.04)
        pred_ax.axis("off")

        overlay_ax.set_title("Overlay")
        overlay_ax.imshow(overlay)
        overlay_ax.axis("off")

        gt_ax.set_title("Ground Truth")
        gt_ax.imshow(gt_mask, cmap="gray", vmin=0, vmax=1)
        gt_ax.axis("off")

    fig.suptitle(f"{dataset_name} fake examples | threshold={dataset_threshold:.4f} ({dataset_threshold_source})", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Single-upload inference using the model loaded by the post-training test cell.
if "model" not in globals() or "device" not in globals():
    raise RuntimeError("Run the post-training test evaluation cell first so the checkpoint and model are loaded.")

from google.colab import files
from PIL import Image
import io
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.dataloaders import _compute_high_pass_fallback
from tools.local_infer_helpers import normalize_image_for_inference, resize_for_inference

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

filename, file_bytes = next(iter(uploaded.items()))
pil_img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
img_np = np.array(pil_img).astype(np.float32) / 255.0
img = torch.from_numpy(img_np).permute(2, 0, 1)
high_pass = _compute_high_pass_fallback(img)

h, w = img.shape[-2], img.shape[-1]
MAX_SIDE_LENGTH = 768
if max(h, w) > MAX_SIDE_LENGTH:
    scale_factor = MAX_SIDE_LENGTH / max(h, w)
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
else:
    new_h, new_w = h, w

target_h = int(math.ceil(new_h / 32) * 32)
target_w = int(math.ceil(new_w / 32) * 32)
if (target_h, target_w) != (h, w):
    img = F.interpolate(img.unsqueeze(0), size=(target_h, target_w), mode="bilinear", align_corners=False).squeeze(0)
    high_pass = F.interpolate(high_pass.unsqueeze(0), size=(target_h, target_w), mode="bilinear", align_corners=False).squeeze(0)

img, _, high_pass = resize_for_inference(img, mask=None, high_pass=high_pass, max_short_side=INFER_MAX_SHORT_SIDE)
x = normalize_image_for_inference(img, normalization_mode=INFER_NORMALIZATION).unsqueeze(0).to(device)
hp = high_pass.unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(x, target_size=img.shape[-2:], high_pass=hp)[-1]
    pred_prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

active_threshold, active_threshold_source = resolve_infer_threshold("test", None)
img_show = img.permute(1, 2, 0).cpu().numpy()
pred_bin = (pred_prob >= float(active_threshold)).astype(np.float32)

overlay = img_show.copy()
overlay[..., 0] = np.clip(overlay[..., 0] + 0.5 * pred_bin, 0, 1)
overlay[..., 1] = np.clip(overlay[..., 1] * (1.0 - 0.35 * pred_bin), 0, 1)
overlay[..., 2] = np.clip(overlay[..., 2] * (1.0 - 0.35 * pred_bin), 0, 1)

print(f"Uploaded: {filename}")
print("Input tensor shape:", tuple(x.shape))
print("Inference threshold:", active_threshold)
print("Inference threshold source:", active_threshold_source)
print("Inference normalization:", INFER_NORMALIZATION)
print("Inference max_short_side:", INFER_MAX_SHORT_SIDE)
print("Prediction stats min/mean/max:", float(pred_prob.min()), float(pred_prob.mean()), float(pred_prob.max()))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("Uploaded Photo")
plt.imshow(img_show)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Prediction")
plt.imshow(pred_prob, cmap="magma", vmin=0, vmax=1)
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Overlay")
plt.imshow(overlay)
plt.axis("off")

plt.tight_layout()
plt.show()